## 🛠️ Data Engineering & Challenges (Lessons Learned)

During the transformation of the raw dataset (`olympics.csv`) into a relational Star Schema for PostgreSQL, several data quality and schema challenges were encountered and resolved within the ETL pipeline:

### 1. Handling Missing Values & Data Types (`Int64` vs. `float`)
* **Issue:** Due to missing values (`NaN`) in columns like `age`, `height`, and `weight`, Pandas automatically cast these fields to `float` (e.g., `180.0`). This caused data type mismatch errors when importing into PostgreSQL (`SMALLINT` / `INTEGER`). Standard NumPy `int` casting also failed because it does not support null values.
* **Solution:** Used Pandas' nullable integer type (**`Int64`**), which preserves true integer values while allowing `NaN` entries.
* **Validation:** Cross-checked row counts of the transformed tables against the original CSV file to ensure no records were lost during type conversion.

### 2. Primary & Foreign Key Integrity (Data Validation)
* **Issue:** Because athletes compete in multiple events and initial joins were performed on names, duplicate IDs were generated. This prevented the fact table (`fact_medal_olympics`) from being created properly with primary key constraints in **pgAdmin**.
* **Solution:** 
  * Properly deduplicated dimension tables (`drop_duplicates()`), specifically ensuring unique country codes (`noc`) and unique athlete combinations (`name` + `sex`).
  * Regenerated a clean, sequential primary key (`id`) for the fact table after all joins and deduplication steps were completed.

### 3. ETL & Database Schema Consistency
* **Standardized Naming:** Converted all column names and categorical strings (such as `noc` and `continent`) to lowercase (`snake_case`) to avoid case-sensitivity issues in PostgreSQL queries.
* **Schema Alignment:** Synchronized column names (`id_co`, `id_ath`, `id_saol`) and data types (`medal_enum`) and reordered columns prior to CSV export to match the PostgreSQL/DrawSQL schema 1:1.
DrawSQL/PostgreSQL schema.

In [1]:
import pandas as pd
import numpy as  np


In [ ]:

df = pd.read_csv("./olympics.csv", sep=";")
df1 = df[df["Year"] >= 1960].copy() # .copy() sorgt dafür, dass df1 eine echte eigene Tabelle ist

df1.columns = df1.columns.str.lower()
df1['noc'] = df1['noc'].str.lower()
dim_country = df1[["noc", "country"]].drop_duplicates(subset=['noc']).reset_index(drop=True)
dim_country["id_co"] = range(1, len(dim_country) + 1)

dim_country['continent']='other'

latam_nocs = ['arg', 'bra', 'mex', 'col', 'chi', 'per', 'ven', 'cub', 'jam', 'nca', 'pur', 'esa', 'ecu','bol']
africa_nocs = ['ken', 'ngr', 'egy', 'rsa', 'eth', 'mar', 'alg', 'tun', 'gha', 'cmr', 'lba','ang', 'mtn','tog','lbr', 'mad', 'caf','cod','sen', 'bur','rho', 'zim', 'uga',]


dim_country.loc[dim_country['noc'].str.lower().isin(latam_nocs), 'continent'] = 'latin america'
dim_country.loc[dim_country['noc'].str.lower().isin(africa_nocs), 'continent'] = 'africa'

dim_athlete = df1[["name", "sex"]].drop_duplicates().reset_index(drop=True)
dim_athlete["id_ath"] = range(1, len(dim_athlete) + 1)

dim_olympics = df1[["sport", "event"]].drop_duplicates().reset_index(drop=True)
dim_olympics["id_saol"] = range(1, len(dim_olympics) + 1)

df1 = pd.merge(df1, dim_country[['noc', 'id_co']], on='noc', how='left')
df1 = pd.merge(df1, dim_athlete[['name', 'sex', 'id_ath']], on=['name','sex'], how='left')
df1 = pd.merge(df1, dim_olympics[['sport', 'event', 'id_saol']], on=['sport', 'event'], how='left')
fact_medal_olympics = df1[["age", "height", "weight", "year", "medal", "id_co", "id_ath", "id_saol"]].drop_duplicates().reset_index(drop=True)

fact_medal_olympics['height'] = fact_medal_olympics['height'].astype('Int64')
fact_medal_olympics['weight'] = fact_medal_olympics['weight'].astype('Int64')
fact_medal_olympics['age'] = fact_medal_olympics['age'].astype('Int64')
fact_medal_olympics['year'] = fact_medal_olympics['year'].astype('Int64')
#Er hat die Werte von Weight und Height in floats geändert, was natürlich Fehler beim einfügen warf. Mit int ging es auch nicht, weil Numpy keine leeren zeilen mag, deswegen 'In64'. Ich habe dann die Eintragszahl mit der csv verglichen so konnte ich das validieren.

fact_medal_olympics['id'] = range(1, len(fact_medal_olympics)+1)#Großes Problem da die Joins über Namen waren. Es waren mehrere IDs vergeben(wenn man mehrmals an Wettkämpfen teilnahm) daher konnte er die Faktentabelle im PG ADmin nicht ordentlich erstellen. Data Validating und so :). So habe ich den Index neuerstellt und das Problem gelöst.



dim_country = dim_country[["id_co", "noc", "country", "continent"]]
dim_athlete = dim_athlete[["id_ath", "name", "sex"]]
dim_olympics = dim_olympics[["id_saol", "sport", "event"]]
fact_medal_olympics = fact_medal_olympics[["id", "id_co", "id_ath", "id_saol", "age", "height", "weight", "year", "medal"]]

